<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/notebooks/04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Directories
SPLIT_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/splits"
WINDOW_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"
OUTPUT_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"

NEG_RATIO = 10.0

# Scalable split handling
file_map = {
    "train": "train_balanced.csv",
    #"val": "val_windows.csv"
    "val": "val_sliding_windows_step10.csv"
}

for split_name, filename in file_map.items():

    file_path = os.path.join(WINDOW_DIR, filename)

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue

    df = pd.read_csv(file_path)

    print(f"\nSplit: {split_name}")
    print("Rows:", len(df))
    print("Windows:", df["window_id"].nunique())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Split: train
Rows: 5520
Windows: 92

Split: val
Rows: 3230820
Windows: 53847


In [3]:
def compute_features(window_df):
    p = window_df["PRESSURE"].values

    features = {}

    # Trend
    x = np.arange(len(p))
    slope = np.polyfit(x, p, 1)[0]
    features["slope"] = slope

    # Pressure drop
    features["pressure_drop"] = p[0] - p.min()
    features["min_position"] = np.argmin(p) / len(p)

    # Statistics
    features["mean"] = p.mean()
    features["std"] = p.std()
    features["range"] = p.max() - p.min()

    # Label
    features["label"] = window_df["label"].iloc[0]

    return features


In [4]:
feature_rows = []

for window_id, window_data in df.groupby("window_id"):
    features = compute_features(window_data)
    feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)


In [5]:
for split_name in ["train", "val"]:

    file_map = {
        "train": "train_balanced.csv",
        "val": "val_windows.csv"
    }

    input_file = os.path.join(WINDOW_DIR, file_map[split_name])
    df = pd.read_csv(input_file)

    # --- compute features here ---
    feature_rows = []

    for window_id, group in df.groupby("window_id"):
        feats = compute_features(group)
        feats["window_id"] = window_id
        feats["label"] = group["label"].iloc[0]
        feature_rows.append(feats)

    features_df = pd.DataFrame(feature_rows)

    output_file = os.path.join(WINDOW_DIR, f"{split_name}_features.csv")

    if os.path.exists(output_file):
        os.remove(output_file)

    features_df.to_csv(output_file, index=False)

    print("Saved:", f"{split_name}_features.csv")


Saved: train_features.csv
Saved: val_features.csv
